# Clustering baselines

These are the raw-data and two-stage experiments from `mnist_basic`, `test_nmf` and `mnist_vae_kmeans`. First learn the representation, then keep it fixed while clustering. Joint models are in `mnist.ipynb` and `deep_clustering.ipynb`.

Run the cells from top to bottom. The default is a short MNIST experiment, not a reproduction of the thesis tables.


In [ ]:
%run ./functions.ipynb
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import kmeans_plusplus
from IPython.display import display


In [ ]:
dataset_name = 'mnist'       # 'digits', 'fashion_mnist', 'reuters10k'
n_samples = 2000             # None uses the complete selected dataset
split = 'all'                # transductive clustering: train and test images
n_clusters = 10
seeds = [42]                 # e.g. list(range(10)) for repeated experiments
pretrain_epochs = 10
nmf_iter = 200
mf_epochs = 100
batch_size = 256
hidden_dims = [128, 64]      # thesis FC layers: [500, 500, 2000]
latent_dim = 10
nmf_loss = 'kullback-leibler' # or 'frobenius'

# Use the same data rows for every method and training seed.
X, y = load_data(dataset_name, n_samples=n_samples, seed=42, split=split)
X_np = X.numpy()
print(dataset_name, X.shape)
records = []
representations = {}
trained_models = {}


## k-means, GMM and NMF

`kmeans_full` initializes both NMF factors using fitted k-means centers and its one-hot assignments. `kmeans_centers` uses k-means++ seeds for the centers and random assignments. These assignments are inferred from the data, not from the class labels. `random` and `nndsvdar` retain the other initializations tried in the notebooks.


In [ ]:
def fit_nmf(X, n_clusters, init='random', n_iter=200, seed=42,
            beta_loss='kullback-leibler'):
    X = np.asarray(X, dtype=np.float32)
    if X.min() < 0:
        raise ValueError('NMF needs nonnegative data. Scale fixed features first.')
    custom = init in ['kmeans_full', 'kmeans_centers']
    model = NMF(n_components=n_clusters, init='custom' if custom else init,
                solver='mu', beta_loss=beta_loss, max_iter=n_iter,
                random_state=seed)
    if custom:
        rng = np.random.default_rng(seed)
        if init == 'kmeans_full':
            km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit(X)
            C = km.cluster_centers_.copy()
            Y = np.eye(n_clusters, dtype=X.dtype)[km.labels_]
        else:
            C, _ = kmeans_plusplus(X, n_clusters=n_clusters, random_state=seed)
            Y = rng.uniform(.1, 1., (len(X), n_clusters)).astype(X.dtype)
        Y = model.fit_transform(X, W=np.ascontiguousarray(Y + 1e-6),
                               H=np.ascontiguousarray(C + 1e-6))
    else:
        Y = model.fit_transform(X)
    return Y, model.components_, model


for seed in seeds:
    predictions = {
        'k-means': KMeans(n_clusters=n_clusters, n_init=10,
                         random_state=seed).fit_predict(X_np),
        'GMM': GaussianMixture(n_components=n_clusters, covariance_type='diag',
                               n_init=1, random_state=seed).fit_predict(X_np)
    }
    for init in ['random', 'kmeans_centers', 'kmeans_full', 'nndsvdar']:
        Y, C, _ = fit_nmf(X_np, n_clusters, init, nmf_iter, seed, nmf_loss)
        predictions['NMF / ' + init] = Y.argmax(1)
    for method, pred in predictions.items():
        records.append({'method': method, 'seed': seed, **score_dict(y, pred)})

classical_scores = pd.DataFrame(records)
display(classical_scores)


## AE / VAE followed by clustering

The AE uses a linear decoder and MSE. The VAE uses BCE for images and MSE for TF-IDF features. K-means receives the original latent coordinates. Only the fixed embeddings sent to nonnegative NMF are MinMax-scaled; this transformation is outside training.

For the VAE we cluster the posterior mean, not a fresh sample.


In [ ]:
for kind in ['ae', 'vae']:
    for seed in seeds:
        seed_everything(seed)
        dimensions = [X.shape[1], *hidden_dims, latent_dim]
        model = (AutoEncoder(dimensions, final_activation=nn.Identity()) if kind == 'ae'
                 else VariationalAutoEncoder(dimensions)).to(device)
        reconstruction = 'mse' if kind == 'ae' or dataset_name == 'reuters10k' else 'bce'
        history = pre_train(X, model, epochs=pretrain_epochs,
                            batch_size=batch_size, loss=reconstruction, seed=seed)
        Z = vae_encode(X, model, batch_size)
        representations[(kind, seed)] = Z
        trained_models[(kind, seed)] = model
        pred = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit_predict(Z)
        records.append({'method': kind.upper() + ' + k-means', 'seed': seed,
                        **score_dict(y, pred)})
        Z_nonnegative = np.maximum(MinMaxScaler().fit_transform(Z), 0).astype(np.float32)
        for init in ['random', 'kmeans_full', 'nndsvdar']:
            Y, C, _ = fit_nmf(Z_nonnegative, n_clusters, init, nmf_iter, seed, nmf_loss)
            records.append({'method': kind.upper() + ' + NMF / ' + init,
                            'seed': seed, **score_dict(y, Y.argmax(1))})

display(pd.DataFrame(records))


## Signed Semi-NMF with fixed features

Here $Z \approx YC$, with $Y \geq 0$ and signed $C$. Alternating least squares updates $C$; the multiplicative update changes $Y$. This is separate from the binary assignment model below. No MinMax transformation is needed.


In [ ]:
def semi_nmf_fixed(Z, n_clusters, n_iter=200, seed=42):
    Z = np.asarray(Z, dtype=np.float64)
    rng = np.random.default_rng(seed)
    Y = rng.uniform(.1, 1., (len(Z), n_clusters))
    history = []
    for _ in range(n_iter):
        C = np.linalg.lstsq(Y, Z, rcond=None)[0]
        A = Z @ C.T
        B = C @ C.T
        A_pos, A_neg = np.maximum(A, 0), np.maximum(-A, 0)
        B_pos, B_neg = np.maximum(B, 0), np.maximum(-B, 0)
        Y *= np.sqrt(np.maximum(A_pos + Y @ B_neg, 1e-12) /
                     np.maximum(A_neg + Y @ B_pos, 1e-12))
        history.append(float(np.mean((Z - Y @ C) ** 2)))
    C = np.linalg.lstsq(Y, Z, rcond=None)[0]
    return Y, C, history


for kind in ['ae', 'vae']:
    for seed in seeds:
        Z = representations[(kind, seed)]
        Y, C, history = semi_nmf_fixed(Z, n_clusters, nmf_iter, seed)
        records.append({'method': kind.upper() + ' + Semi-NMF', 'seed': seed,
                        **score_dict(y, Y.argmax(1))})

display(pd.DataFrame(records).tail(2 * len(seeds)))


## Fixed-feature MF with binary updates

The original `test_nmf` compared random, center-only and full k-means initialization, with a warm-up before applying the binary update. This function keeps those choices. `legacy` retains the accumulated update direction; `improved` uses the row-adaptive PRIMP update directly.

The strength follows a fixed schedule. An intermediate notebook adjusted it after looking at accuracy; that label-dependent adjustment is omitted here. Training never reads `y`.


In [ ]:
def test_nmf_clustering(Z, n_clusters=10, n_epochs=100, lr=1e-2,
                        init='kmeans_full', temperature=.1, warm_start_epochs=10,
                        kappa=1e-4, double_every=50, binary_mode='improved',
                        seed=42):
    seed_everything(seed)
    Z = torch.as_tensor(Z, dtype=torch.float32, device=device)
    rng = np.random.default_rng(seed)
    Y_init = rng.uniform(0, 1, (len(Z), n_clusters)).astype(np.float32)
    if init == 'random':
        C_init = rng.normal(0, .1, (n_clusters, Z.shape[1])).astype(np.float32)
    elif init == 'kmeans_centers':
        C_init, _ = kmeans_plusplus(Z.cpu().numpy(), n_clusters=n_clusters,
                                    random_state=seed)
    elif init in ['kmeans_full', 'kmeans_soft']:
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit(Z.cpu().numpy())
        C_init = km.cluster_centers_
        Y_init = np.eye(n_clusters, dtype=np.float32)[km.labels_]
        if init == 'kmeans_soft':
            if temperature <= 0:
                raise ValueError('temperature must be positive')
            # Retain the soft initialization used in the exploratory notebook.
            values = torch.tensor(Y_init) + torch.randn_like(torch.tensor(Y_init)) * temperature
            values = torch.sigmoid(values / temperature)
            Y_init = (values / values.max(1, keepdim=True).values).numpy()
    else:
        raise ValueError('Unknown initialization: ' + init)
    mf = MatrixFactorizationBinaryModel(len(Z), n_clusters, Z.shape[1],
                                         Y_init=Y_init, C_init=C_init).to(device)
    optimizerC = torch.optim.Adam([mf.C], lr=lr)
    optimizerY = torch.optim.Adam([mf.Y], lr=lr)
    accumulated = torch.zeros_like(mf.Y)
    history = {'mf_loss': [], 'binary_error': [], 'onehot_rate': []}
    for epoch in range(n_epochs):
        optimizerC.zero_grad()
        loss = F.mse_loss(mf.Y.detach() @ mf.C, Z)
        loss.backward()
        optimizerC.step()
        optimizerY.zero_grad()
        loss = F.mse_loss(mf.Y @ mf.C.detach(), Z)
        loss.backward()
        optimizerY.step()
        with torch.no_grad():
            mf.Y.clamp_(0, 1)
            strength = kappa * (2 ** (epoch // double_every) if double_every else 1)
            if epoch >= warm_start_epochs:
                if binary_mode == 'improved':
                    mf.Y.copy_(improved_primp(mf.Y, strength))
                elif binary_mode == 'primp':
                    mf.Y.copy_(prox_primp(mf.Y, strength))
                elif binary_mode == 'legacy':
                    root = mf.Y.topk(2, dim=1).values.mean(1, keepdim=True)
                    accumulated.add_(strength * (mf.Y - root))
                    mf.Y.add_(accumulated, alpha=lr).clamp_(0, 1)
                elif binary_mode != 'none':
                    raise ValueError('Unknown binary mode: ' + binary_mode)
            history['mf_loss'].append(F.mse_loss(mf.Y @ mf.C, Z).item())
            history['binary_error'].append(torch.minimum(mf.Y, 1 - mf.Y).mean().item())
            onehot = F.one_hot(mf.Y.argmax(1), n_clusters).float()
            history['onehot_rate'].append(((mf.Y - onehot).abs().max(1).values < 1e-3).float().mean().item())
    return {'mf': mf, 'pred': mf.Y.detach().argmax(1).cpu().numpy(), 'history': history}


fixed_runs = {}
for seed in seeds:
    Z = representations[('vae', seed)]
    for binary_mode in ['none', 'improved']:
        result = test_nmf_clustering(Z, n_clusters, n_epochs=mf_epochs, seed=seed,
                                     binary_mode=binary_mode)
        fixed_runs[(binary_mode, seed)] = result
        records.append({'method': 'VAE + fixed MF / ' + binary_mode, 'seed': seed,
                        **score_dict(y, result['pred'])})

fixed_history = fixed_runs[('improved', seeds[-1])]['history']
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, name in zip(axes, ['mf_loss', 'binary_error', 'onehot_rate']):
    ax.plot(fixed_history[name])
    ax.set(xlabel='Epoch', title=name)
fig.tight_layout()
plt.show()


## Summary and optional initialization comparison

Each row below is one complete run. The mean and standard deviation summarize runs; there is no selection of a different best seed for each metric. A single seed has an undefined sample standard deviation.


In [ ]:
scores = pd.DataFrame(records)
display(scores)
display(scores.groupby('method')[['ACC', 'NMI', 'ARI', 'F1', 'Purity']].agg(['mean', 'std']))

save_results = False
if save_results:
    Path('results').mkdir(exist_ok=True)
    scores.to_csv('results/baselines.csv', index=False)


In [ ]:
compare_initializations = False
if compare_initializations:
    comparison = []
    Z = representations[('vae', seeds[0])]
    for init in ['random', 'kmeans_centers', 'kmeans_full', 'kmeans_soft']:
        for binary_mode in ['none', 'legacy', 'primp', 'improved']:
            result = test_nmf_clustering(Z, n_clusters, n_epochs=mf_epochs,
                                         init=init, binary_mode=binary_mode, seed=seeds[0])
            comparison.append({'init': init, 'binary': binary_mode,
                               **score_dict(y, result['pred'])})
    display(pd.DataFrame(comparison))
